# Hot swap the live model in the Serve app

This notebook changes which OpenFold3 checkpoint answers live predictions in a running Anyscale
Service, with no restart and no new cluster. At each step it sends the same molecule and compares the
pose with the molecule's crystal structure.

**The demo:** EV-A71 2A compound x7259a, a potent late follow-on (pKD 7.9). None of the models trained
on it. All steps use the same model, OpenFold3-p2, with different weights:

| Step | Checkpoint | Training on EV-A71 2A | x7259a, published RMSD to the crystal |
|---|---|---|---|
| 1. Base model | `of3-p2-145k.pt` | None | 5.9 Å, wrong in 5 of 5 seeds |
| 2. Our fine-tune | `of3_ev71_ft.pt`, started from `of3-p2-145k.pt` | Ray Train on the fragment screen | Not measured yet |
| 3. Public fine-tune | `of3p2-ft-ev2a.ckpt` (OpenBind) | 79 fragment structures, 8 GPUs, about 1,000 steps, 32 diffusion samples per step | 0.5 Å, correct in 5 of 5 seeds |

Published values: the OpenBind benchmark, top-ranked pose of 5 seeds. On all EV-A71 2A follow-ons,
step 1 places 12% of poses within 2 Å and step 3 places 63%. Step 2 is skipped until `OURS` is set.

**How the swap works**

- `FineTunedLive` takes `{"version", "checkpoint", "name", "legacy"}` as its Ray Serve `user_config`.
  `legacy: true` runs the checkpoint with OpenFold3 0.4.3, which every OpenFold3-p2 checkpoint needs.
  The `Viewer` takes `{"pose_from"}`. Here the pose always comes from `FineTunedLive`, and Boltz-2
  adds the affinity.
- `anyscale.service.deploy(..., in_place=True)` with a config that changes only `user_config` calls
  `reconfigure()` on the running replicas. Ray Serve does not restart them.
- Each version keeps its poses in its own cache folder. Do not name a version `base`: that folder
  holds the poses of the released fine-tune.

**Before you start**

- Run this notebook in a Workspace, from the repository folder. It deploys `service.yaml` from here.
- The data and the checkpoints must be in `/mnt/shared_storage/aiddd`. If `of3-p2-145k.pt` is
  missing, run `anyscale job submit -f job.yaml -- python scripts/setup_data.py`.
- A new molecule takes one to three minutes per step. Run all steps once before the session, so each
  step returns from its cache in seconds.
- Do not change files in this folder between the first deploy and the swaps. A code change makes the
  update restart every replica. Notebooks are excluded from the upload, so editing this one is safe.

In [ ]:
import json, os, time
from pathlib import Path

import anyscale
import pandas as pd
import requests
from anyscale.service.models import ServiceConfig

SHARED = Path(os.environ.get("AIDDD_SHARED", "/mnt/shared_storage/aiddd"))
CKPTS = SHARED / "of3_cache"
assert (SHARED / "results" / "ev71-2a").exists(), f"missing {SHARED}/results: copy the data first"
assert (CKPTS / "of3-p2-145k.pt").exists(), "missing of3-p2-145k.pt: run scripts/setup_data.py"

# Your own Service. "aiddd-viewer" is the shared workshop Service: never use it for practice.
SERVICE_NAME = "aiddd-viewer-geoff"
assert SERVICE_NAME != "aiddd-viewer", "pick your own Service name for practice runs"

# x7259a: the base model is wrong, the public fine-tune is right. Backups: x7570a, x7056a.
SMILES = "CSc1ncnc(N2CCC[C@@H]2C2CC2)c1CC(=O)N1CCC[C@H](C(N)=O)C1"

# Step 2: our Ray Train fine-tune of of3-p2-145k.pt. None skips the step.
OURS = None   # e.g. SHARED / "finetune/runs/<run>/checkpoint_<n>/of3_ev71_ft.pt"
assert OURS is None or Path(OURS).exists(), f"missing {OURS}"


def step(version: str, name: str, checkpoint: Path) -> dict:
    """user_config per deployment for one step. All are OF3-p2 checkpoints: legacy OpenFold3."""
    return {"Viewer": {"pose_from": "FineTunedLive"},
            "FineTunedLive": {"version": version, "name": name, "checkpoint": str(checkpoint), "legacy": True}}


STEPS = {"p2-145k": step("p2-145k", "OF3-p2 145k, not fine-tuned", CKPTS / "of3-p2-145k.pt")}
if OURS:
    STEPS["ours-145k"] = step("ours-145k", "OF3-p2 145k + our Ray Train fine-tune", OURS)
STEPS["public-ft"] = step("public-ft", "OF3-p2 + public fine-tune (8 GPUs, 1,000 steps)",
                          CKPTS / "of3p2-ft-ev2a.ckpt")
list(STEPS)

## Helpers

`service_config()` is `service.yaml` with your Service name and a `user_config` for each listed
deployment. `predict()` calls the Viewer's `/predict` endpoint, the same one the web page uses.
`swap_to()` makes one in-place update, then sends the molecule until the pose comes from the new
version.

In [ ]:
def service_config(user_configs: dict) -> ServiceConfig:
    """service.yaml, renamed, with {deployment name: user_config} overrides."""
    cfg = ServiceConfig.from_yaml("service.yaml", name=SERVICE_NAME)
    deployments = [{"name": d, "user_config": uc} for d, uc in user_configs.items()]
    return cfg.options(applications=[{**app, "deployments": deployments} for app in cfg.applications])


def predict(smiles: str) -> dict:
    s = anyscale.service.status(name=SERVICE_NAME)
    headers = {"Authorization": f"Bearer {s.query_auth_token}"} if s.query_auth_token else {}
    r = requests.post(s.query_url.rstrip("/") + "/predict", json={"smiles": smiles},
                      headers=headers, timeout=600)
    return r.json()


def summary(out: dict) -> dict:
    """The model behind each part of one /predict response, and the main numbers."""
    models, key, m = out.get("models") or {}, out.get("answer_key") or {}, out.get("metrics", {})
    return {"pose model": f"{models.get('pose', {}).get('name')} ({models.get('pose', {}).get('version')})",
            "affinity model": f"{models.get('affinity', {}).get('name')} ({models.get('affinity', {}).get('version')})",
            "crystal": key.get("complex"), "ligand RMSD (Å)": key.get("lig_rmsd"),
            "pose ipTM": out.get("pose_iptm"), "affinity (log10 IC50, µM)": m.get("affinity_pred_value"),
            "status": out.get("status"), "seconds": out.get("seconds")}


responses = {}


def swap_to(version: str) -> dict:
    """In-place update to one step, then the molecule until the pose comes from that version."""
    if version not in STEPS:
        return {"skipped": f"{version}: no checkpoint yet"}
    t0 = time.time()
    anyscale.service.deploy(service_config(STEPS[version]), in_place=True, name=SERVICE_NAME)
    anyscale.service.wait(name=SERVICE_NAME, state="RUNNING", timeout_s=600)
    for _ in range(30):
        out = predict(SMILES)
        if (out.get("models") or {}).get("pose", {}).get("version") == version:
            break
        time.sleep(5)
    else:
        raise TimeoutError(f"the pose still does not come from {version}")
    print(f"swap and prediction took {time.time() - t0:.0f} s")
    responses[version] = out
    return summary(out)

## 1. Start the Service at the base model

The first deploy starts a new cluster with two L4 nodes, which takes about 5 to 10 minutes. If the
Service is already running, this cell only resets it to the base model, in place.

In [ ]:
try:
    running = str(anyscale.service.status(name=SERVICE_NAME).state) == "RUNNING"
except Exception:   # no Service with this name yet
    running = False
anyscale.service.deploy(service_config(STEPS["p2-145k"]), in_place=running, name=SERVICE_NAME)
anyscale.service.wait(name=SERVICE_NAME, state="RUNNING", timeout_s=1800)
print(anyscale.service.status(name=SERVICE_NAME).query_url)

## 2. Base model: OpenFold3-p2, not fine-tuned

A general model. It has never seen EV-A71 2A ligands bound to this pocket.

In [ ]:
swap_to("p2-145k")

## 3. Our fine-tune, with Ray Train

The same base model, fine-tuned on the EV-A71 2A fragment screen with `finetune_ray_train.py`.
Skipped until `OURS` is set.

In [ ]:
swap_to("ours-145k")

## 4. The public fine-tune

The OpenBind team's fine-tune of the same base model: 8 GPUs, about 1,000 steps, 32 diffusion samples
per step.

In [ ]:
swap_to("public-ft")

## 5. All steps side by side

In [ ]:
pd.DataFrame({v: summary(o) for v, o in responses.items()})

The full response metadata of the last step, as the web page receives it:

In [ ]:
last = list(responses.values())[-1]
print(json.dumps({k: last.get(k) for k in ("models", "answer_key", "pose_model", "pose_iptm")}, indent=2))

## 6. Swap back

The same in-place update, back to the base model. It waits until the update is done, so a rerun
from the top does not start while this update is still in progress.

In [ ]:
anyscale.service.deploy(service_config(STEPS["p2-145k"]), in_place=True, name=SERVICE_NAME)
anyscale.service.wait(name=SERVICE_NAME, state="RUNNING", timeout_s=600)

## 7. Stop the Service

The Service keeps two L4 nodes running until you stop it.

In [ ]:
# anyscale.service.terminate(name=SERVICE_NAME)